# JEI Stock Radar — Colab 風險與資料驗證
讀取 GitHub main 的 `remote/system.json`，檢查日期、風險訊號與盤中行情狀態。**不會發送 LINE，也不會修改正式系統。**
執行前可在 Colab 選「執行階段 → 全部執行」。

In [ ]:
import json, urllib.request, datetime, csv
from zoneinfo import ZoneInfo
URL='https://raw.githubusercontent.com/alwaysjei1234-sys/JEI-Stock-Radar/main/remote/system.json'
with urllib.request.urlopen(URL,timeout=20) as response:
    feed=json.load(response)
assert isinstance(feed,dict), 'feed must be an object'
print('Data date:',feed.get('data_date'),'Updated:',feed.get('updated_at'))


In [ ]:
risk=feed.get('risk') or {}
market=feed.get('market') or {}
issues=[]
if not feed.get('data_date'): issues.append('缺少 data_date')
if risk.get('score') is None: issues.append('缺少 risk.score')
if not isinstance(risk.get('reasons'),list): issues.append('缺少 risk.reasons')
if market.get('live_valid') is False and market.get('live_count',0)>0: issues.append('盤中數量非零但 live_valid=false，需人工確認')
now=datetime.datetime.now(ZoneInfo('Asia/Taipei'))
try:
    age=(now.date()-datetime.date.fromisoformat(feed['data_date'])).days
    if age<0: issues.append('資料日期晚於台北今日')
    if age>4: issues.append(f'資料日期距今 {age} 天，可能過期（假日需人工確認）')
except (KeyError,ValueError,TypeError): issues.append('data_date 無法解析')
print('Risk:',risk.get('level'),risk.get('score'))
print('Market:',market.get('status'),'live_valid:',market.get('live_valid'))
print('Issues:',issues or '無基本結構異常')
if risk.get('level')=='red' or (isinstance(risk.get('score'),(int,float)) and risk['score']>=80):
    print('ALERT CANDIDATE: 高風險，應檢查正式通知管線；本筆記本不代表已推播')


In [ ]:
from pathlib import Path
out=Path('jei_colab_audit.csv')
with out.open('w',newline='',encoding='utf-8-sig') as f:
    writer=csv.writer(f)
    writer.writerow(['checked_at_taipei','data_date','risk_level','risk_score','market_status','live_valid','issues'])
    writer.writerow([now.isoformat(),feed.get('data_date'),risk.get('level'),risk.get('score'),market.get('status'),market.get('live_valid'),' | '.join(issues)])
print('Saved:',out)


## 正式系統問題診斷（唯讀）
檢查公開 GitHub Actions 執行結果、工作流程清單及遠端 JSON HTTP 狀態。私人 LINE token 不讀取、不輸出；無法從公開資料判定通知是否送達。


In [ ]:
import urllib.error
API='https://api.github.com/repos/alwaysjei1234-sys/JEI-Stock-Radar'
def get_json(url):
    req=urllib.request.Request(url,headers={'User-Agent':'JEI-Colab-Audit','Accept':'application/vnd.github+json'})
    try:
        with urllib.request.urlopen(req,timeout=20) as r: return json.load(r),r.status
    except urllib.error.HTTPError as e: return {'error':str(e)},e.code
    except Exception as e: return {'error':str(e)},None
workflows,wstatus=get_json(API+'/actions/workflows?per_page=100')
runs,rstatus=get_json(API+'/actions/runs?per_page=30')
print('Workflow API:',wstatus,'Run API:',rstatus)
for w in workflows.get('workflows',[]): print('WORKFLOW',w.get('name'),w.get('state'),w.get('path'))
for r in runs.get('workflow_runs',[])[:15]: print('RUN',r.get('name'),r.get('status'),r.get('conclusion'),r.get('created_at'),r.get('html_url'))
if wstatus!=200 or rstatus!=200: print('WARNING: GitHub API 無法完整讀取，請檢查限流或權限')


In [ ]:
from urllib.parse import urlparse
targets=['https://raw.githubusercontent.com/alwaysjei1234-sys/JEI-Stock-Radar/main/remote/system.json','https://raw.githubusercontent.com/alwaysjei1234-sys/JEI-Stock-Radar/main/remote/update.json','https://raw.githubusercontent.com/alwaysjei1234-sys/JEI-Stock-Radar/main/remote/index.html']
for url in targets:
    req=urllib.request.Request(url,headers={'User-Agent':'JEI-Colab-Audit'})
    try:
        with urllib.request.urlopen(req,timeout=20) as response:
            body=response.read()
            print(urlparse(url).path,'HTTP',response.status,'bytes',len(body),'type',response.headers.get('Content-Type'))
            if url.endswith('.json'): json.loads(body)
    except Exception as e: print('FAIL',urlparse(url).path,type(e).__name__,str(e)[:180])
print('注意：HTTP 正常不代表 Android App 或 LINE 通知正常')


In [ ]:
import json
report={'checked_at_taipei':now.isoformat(),'feed_date':feed.get('data_date'),'risk_level':risk.get('level'),'risk_score':risk.get('score'),'feed_issues':issues,'workflow_api_status':wstatus,'runs_api_status':rstatus,'recent_runs':[{'name':r.get('name'),'status':r.get('status'),'conclusion':r.get('conclusion'),'url':r.get('html_url')} for r in runs.get('workflow_runs',[])[:15]],'note':'No LINE delivery or Android device test performed'}
with open('jei_diagnostics.json','w',encoding='utf-8') as f: json.dump(report,f,ensure_ascii=False,indent=2)
print('Saved jei_diagnostics.json — 可下載後傳回 ChatGPT 分析')


## 保存診斷報告至 Google Drive
首次執行會要求 Google 帳號授權。之後每次執行本筆記本仍可能需要重新掛載，但不必手動下載 JSON。報告只包含公開系統診斷資訊，請勿加入私人 token。


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
dest=Path('/content/drive/MyDrive/JEI-Stock-Radar/diagnostics')
dest.mkdir(parents=True,exist_ok=True)
source=Path('/content/jei_diagnostics.json')
if not source.exists(): raise FileNotFoundError('請先執行前面的診斷程式區塊')
target=dest/'jei_diagnostics_latest.json'
shutil.copy2(source,target)
archive=dest/f'jei_diagnostics_{now.strftime("%Y%m%d_%H%M%S")}.json'
shutil.copy2(source,archive)
print('Saved latest:',target)
print('Saved archive:',archive)
